# EVA — Colab Training (T4)

Google Drive → T4 GPU (16 GB) → обучение B=32/ML=256 → автосинк на Drive.

**Перед запуском**: загрузите на `MyDrive/FCF/`:
- `full_corpus_ru.txt` (172 MB) — **обязательно** (кодируется автоматически)
- `real_data/full_corpus_encoded.npy` (425 MB) — опционально (кодирование быстрее)
- `checkpoints/full_latest.pt` — опционально (resume)
- `checkpoints/full_best.pt` / `trajectory_store_full.pkl` — опционально

In [ ]:
import os
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
print('Drive ready')

In [ ]:
import shutil
REPO = '/content/FCF'
DRIVE = '/content/drive/MyDrive/FCF'
NPY_GDRIVE_ID = '1D6GF9IPmXXvABdPWCstTcsnCiomPG6bh'

if not os.path.exists(REPO):
    !git clone https://github.com/BlackCatSpb/FCF.git {REPO}
else:
    !cd {REPO} && git pull
os.chdir(REPO)

npy_dst = f'{REPO}/real_data/full_corpus_encoded.npy'
if not os.path.exists(npy_dst):
    os.makedirs(f'{REPO}/real_data', exist_ok=True)
    # Скачиваем .npy с Google Drive (нужен доступ \"Anyone with the link\")
    !pip install gdown -q
    !gdown {NPY_GDRIVE_ID} -O {npy_dst} --fuzzy 2>&1
    if os.path.exists(npy_dst) and os.path.getsize(npy_dst) > 1e6:
        print(f'Downloaded .npy ({os.path.getsize(npy_dst)//1e6:.0f} MB)')
    else:
        # fallback: ищем на смонтированном Drive
        npy_src = f'{DRIVE}/real_data/full_corpus_encoded.npy'
        if os.path.exists(npy_src):
            shutil.copy2(npy_src, npy_dst)
            print(f'Copied .npy from Drive ({os.path.getsize(npy_dst)//1e6:.0f} MB)')
        else:
            print('Download failed, encoding from .txt...')
            txt_src = f'{DRIVE}/full_corpus_ru.txt'
            if os.path.exists(txt_src):
                shutil.copy2(txt_src, f'{REPO}/real_data/full_corpus_ru.txt')
                !python encode_full_corpus.py
            else:
                raise FileNotFoundError('No data source.\n'
                    '1. Set Drive file permission to \"Anyone with the link\" and retry\n'
                    '2. Or upload full_corpus_ru.txt to MyDrive/FCF/\n'
                    '3. Or mount Drive with the file at FCF/real_data/full_corpus_encoded.npy')

for fn in ['full_latest.pt', 'full_best.pt', 'trajectory_store_full.pkl']:
    src = f'{DRIVE}/checkpoints/{fn}'
    dst = f'{REPO}/checkpoints/symbolic/{fn}'
    if os.path.exists(src) and not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        print(f'Copied {fn}')
print('Data ready')

In [ ]:
!pip install numpy scikit-learn --quiet
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.mem_get_info()[1]/1e9:.1f} GB')

In [ ]:
%%writefile /content/FCF/eva/symbolic/colab_config.py
DEVICE = 'cuda'
STEPS = 100000
LR = 5e-3
B = 32
ML = 256
CKPT = '/content/FCF/checkpoints/symbolic'
DRIVE_DIR = '/content/drive/MyDrive/FCF/checkpoints'

In [ ]:
print('>>> Training on Colab T4...')
!python train_full_corpus.py 2>&1

In [ ]:
import shutil
DRIVE_SYNC = '/content/drive/MyDrive/FCF/checkpoints'
os.makedirs(DRIVE_SYNC, exist_ok=True)
for fn in ['full_latest.pt', 'full_best.pt', 'trajectory_store_full.pkl']:
    src = f'/content/FCF/checkpoints/symbolic/{fn}'
    if os.path.exists(src):
        shutil.copy2(src, f'{DRIVE_SYNC}/{fn}')
        print(f'Synced {fn}')

from google.colab import files
best = f'{DRIVE_SYNC}/full_best.pt'
if os.path.exists(best):
    files.download(best)
elif os.path.exists('/content/FCF/checkpoints/symbolic/full_best.pt'):
    files.download('/content/FCF/checkpoints/symbolic/full_best.pt')